[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/module-3-cleaning-and-exploring/notebook-3.5-trips.ipynb)


# Module 3 — Exploring Trip Data with an LLM (Colab walkthrough)

A hands-on companion to the lecture on **exploring data with an LLM**, run on a small, **synthetic** bike-share **trips** table (already clean and joined).

## What this notebook is for

This notebook is a **preview** of what the lecture will show you. Run it top to bottom — or just read the printed outputs — to get a feel for the work before the lecture explains it: turning a vague curiosity into one specific question at a time, and letting each answer suggest the next. You don't need to follow every line of code yet. The goal is to **picture what we'll be doing**, so when the lecture walks through the *why*, you already have a concrete example in your head. Every step prints a result you can check.

### How to run in Google Colab
1. `Runtime ▸ Run all`. The notebook **reads the dataset straight from this course's public GitHub repo** — nothing to upload, nothing to generate.
2. Everything below reads that CSV and prints a checkable result for each thing the lecture shows.

> ⚠️ **The data is synthetic.** Rows are fabricated to be *plausible* and to match the columns, types, quirks, and numbers used in the Module 3 lecture scripts. It is **not** real Bicing data — real Bicing snapshots are ~370 MB (too big for Colab) and no public trip-level Bicing data exists, which is why this small synthetic stand-in is hosted in the course repo and read directly above.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # headless backend (works in Colab and CI)
import matplotlib.pyplot as plt

# This course's practice data lives in its public GitHub repo. We read it straight
# from there, so the notebook runs the same in Google Colab or anywhere online —
# nothing to upload, nothing to generate.
DATA_URL = ("https://raw.githubusercontent.com/pmarcelino/mobillity-courses/main/"
            "mobillity-univ/module-3-cleaning-and-exploring/data/"
            "barcelona-bicing-synthetic/")


## Exploring Data with LLMs

The data is already clean and joined, so we stop asking *what is this data?* and start
asking *what does it tell us?* — one specific question at a time, each answer shaping the
next.


In [2]:
trips = pd.read_csv(f"{DATA_URL}trips-sample.csv", parse_dates=["start_time"])
trips["hour_of_day"] = trips["start_time"].dt.hour

# Q1: when are peak usage hours?
by_hour = trips.groupby("hour_of_day").size()
print("trips by hour (top 5):")
print(by_hour.sort_values(ascending=False).head(5).to_string())
print("busiest hour:", int(by_hour.idxmax()))

fig, ax = plt.subplots(figsize=(8, 3))
by_hour.plot(kind="bar", ax=ax); ax.set_title("Trips by hour of day"); ax.set_xlabel("hour")
fig.tight_layout(); fig.savefig("fig_3.5_trips_by_hour.png", dpi=90); plt.close(fig)
print("saved:", "fig_3.5_trips_by_hour.png")


trips by hour (top 5):
hour_of_day
18    659
8     646
12    480
17    465
19    451
busiest hour: 18
saved: fig_3.5_trips_by_hour.png


In [3]:
# Q2: which stations dominate at peak hours?
peak_hours = by_hour.sort_values(ascending=False).head(4).index.tolist()
peak = trips[trips["hour_of_day"].isin(peak_hours)]
top_stations = peak.groupby("start_station").size().sort_values(ascending=False).head(5)
print("peak window hours:", sorted(peak_hours))
print("top stations at peak:")
print(top_stations.to_string())


peak window hours: [8, 12, 17, 18]
top stations at peak:
start_station
Gran Via Corts Catalanes    188
Carrer de Arago             157
Roger de Flor               156
Napols                      151
Passeig de Gracia           149


In [4]:
# Q3: is trip duration different on weekends vs weekdays?
trips["is_weekend"] = trips["start_time"].dt.dayofweek >= 5
summary = trips.groupby("is_weekend")["duration_min"].agg(["mean", "median", "std"])
summary.index = ["weekday", "weekend"]
print(summary.round(1).to_string())
wk = trips.loc[~trips["is_weekend"], "duration_min"].mean()
we = trips.loc[trips["is_weekend"], "duration_min"].mean()
print(f"mean duration: weekday {wk:.1f} min vs weekend {we:.1f} min -> weekend trips are longer")

fig, ax = plt.subplots(figsize=(6, 3.2))
trips.boxplot(column="duration_min", by="is_weekend", ax=ax)
ax.set_title("Trip duration: weekday (False) vs weekend (True)"); plt.suptitle("")
fig.tight_layout(); fig.savefig("fig_3.5_weekend_duration.png", dpi=90); plt.close(fig)
print("saved:", "fig_3.5_weekend_duration.png")


         mean  median  std
weekday  11.9    12.0  4.0
weekend  24.2    24.0  8.1
mean duration: weekday 11.9 min vs weekend 24.2 min -> weekend trips are longer
saved: fig_3.5_weekend_duration.png


## Done — trip exploration

The exploration lecture has run on the synthetic trips: trips-by-hour with the commute
peak, the busiest stations inside the peak window, and the weekend-vs-weekday duration
difference. Each answer is the kind of result that shapes the next question. Re-run any
time — the notebook reads the same published data, so the numbers don't change. Then try
the matching exercise yourself.
